# 03 · Train (Kaggle GPU, T4 x2)

Each GPU works through its own queue of experiments, one after another. Checkpoints are uploaded to a private HF model repo every 30 min, so re-running this notebook with the same queues resumes unfinished runs and skips finished ones.

**Setup:** Settings → Accelerator: *GPU T4 x2*, Internet: *On*. Add-ons → Secrets: `HF_TOKEN`. Run with *Save Version → Save & Run All (Commit)*; no cell needs input.

| Session | GPU 0 | GPU 1 | ~Hours |
|---|---|---|---|
| 1 | tr000 → tr010 | tr100 → tr050 | 8.3 |
| 2 | tr030 | tr_only_control → tr030_seed1 | 5.5 |

In [ ]:
QUEUES = [
    ["tr000", "tr010"],  # GPU 0
    ["tr100", "tr050"],  # GPU 1
]
RUN_BENCHMARK = False  # 100-step throughput test before training (~12 min)

HF_USER = "DeerKaraca"
DATA_REPO = f"{HF_USER}/bilingual-slm-data"
RUNS_REPO = f"{HF_USER}/bilingual-slm-runs"
WORK = "/tmp/slm"  # data and checkpoints stay out of /kaggle/working, so they are not saved as notebook output

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/DeerKaraca/bilingual-slm.git
%cd bilingual-slm
!git log --oneline -1
!pip install -q -e ".[data,train]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["tokens/*", "tokenizer.json"], local_dir=f"{WORK}/data")

In [ ]:
import subprocess
import sys


def train_cmd(name, *overrides):
    return [sys.executable, "-m", "slm.train", "--config", f"configs/experiments/{name}.yaml", "--set",
            f"data.tokens_dir={WORK}/data/tokens", f"data.tokenizer={WORK}/data/tokenizer.json", *overrides]


if RUN_BENCHMARK:
    subprocess.run(train_cmd("tr050", f"out_dir={WORK}/bench", "train.total_tokens=26214400", "train.eval_every=50"), check=True)

In [ ]:
import threading
from pathlib import Path

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
exit_codes = {}


def run_queue(gpu, names):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    for name in names:
        with open(LOGS / f"{name}.log", "a") as log:
            cmd = train_cmd(name, f"out_dir={WORK}/runs", f"hub_repo={RUNS_REPO}")
            exit_codes[name] = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env).returncode


threads = [threading.Thread(target=run_queue, args=(gpu, names)) for gpu, names in enumerate(QUEUES)]
for t in threads:
    t.start()

In [ ]:
import time

# Keep the session alive and print the latest log line of each run every 10 minutes.
while any(t.is_alive() for t in threads):
    time.sleep(600)
    for log in sorted(LOGS.glob("*.log")):
        lines = log.read_text().strip().splitlines()
        print(f"{log.stem}: {lines[-1][:200] if lines else '(starting)'}", flush=True)

print("exit codes:", exit_codes)
assert all(code == 0 for code in exit_codes.values()), "a run failed, see /kaggle/working/logs"